In [ ]:
from brian2 import *
import sys
sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')

from neuronModels import *
from ringAttractorTEMP import *
from plottingTools import *
from utils import compute_firing_rate

import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, FloatText, Dropdown
from ipywidgets import interactive as interactive_ipyw

# Simulation parameters
defaultclock.dt = 0.1*ms

In [ ]:
# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=50, 
    max=200, 
    step=10, 
    value=120, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

tau_slider = FloatSlider(
    min=1, 
    max=20, 
    step=1, 
    value=10, 
    description='Tau (ms):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_noise_slider = FloatSlider(
    min=0.1, 
    max=5, 
    step=0.1, 
    value=1, 
    description='Noise Sigma (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.1, 
    value=0, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.1, 
    value=0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

I0_slider = FloatSlider(
    min=10, 
    max=50, 
    step=5, 
    value=30, 
    description='Input Amplitude (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_exc_slider = FloatSlider(
    min=0.1, 
    max=3.0, 
    step=0.1, 
    value=1.0, 
    description='Sigma Excitatory:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_inh_slider = FloatSlider(
    min=1.0, 
    max=5.0, 
    step=0.1, 
    value=3.0, 
    description='Sigma Inhibitory:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

g_exc_slider = FloatSlider(
    min=0.05, 
    max=0.5, 
    step=0.01, 
    value=0.1, 
    description='g Excitatory (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

g_inh_slider = FloatSlider(
    min=-0.5, 
    max=-0.05, 
    step=0.01, 
    value=-0.15, 
    description='g Inhibitory (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

duration_slider = FloatSlider(
    min=1, 
    max=5, 
    step=0.5, 
    value=2, 
    description='Duration (s):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

profile_dropdown = Dropdown(
    options=['mexican_hat', 'gaussian', 'cosine'],
    value='mexican_hat',
    description='Synapse Profile:',
    style={'description_width': '150px'},
)

angles_dropdown = Dropdown(
    options=[ 'degrees', 'radians', 'radians (symbolic)'],
    value='degrees',
    description='Angular Representation:',
    style={'description_width': '150px'},
)

# Create dictionary of widgets
widgets = {
    'num_neurons': num_neurons_slider,
    'tau_val': tau_slider,
    'sigma_noise_val': sigma_noise_slider,
    'stimulus_center': stimulus_center_slider,
    'stimulus_width': stimulus_width_slider,
    'I0_val': I0_slider,
    'sigma_exc_val': sigma_exc_slider,
    'sigma_inh_val': sigma_inh_slider,
    'g_exc_val': g_exc_slider,
    'g_inh_val': g_inh_slider,
    'duration_val': duration_slider,
    'syn_profile': profile_dropdown,
    'ticks_angles': angles_dropdown
}

In [ ]:
def interactive_simulator(num_neurons, tau_val, sigma_noise_val, stimulus_center, stimulus_width, I0_val, 
                          sigma_exc_val, sigma_inh_val, g_exc_val, g_inh_val, duration_val, syn_profile, ticks_angles):
    
    # Clear any previous figures
    plt.close('all')
    
    # Convert slider values to Brian units
    tau = tau_val * ms
    sigma_noise = sigma_noise_val * mV
    V_rest = -70 * mV
    I0 = I0_val * mV
    sim_duration = duration_val*second
    g_exc = g_exc_val * mV
    g_inh = g_inh_val * mV
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))
    

    # Set up neuron model
    neuron_eq = Equations(LIF_xi_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)
    
    # Set up ring attractor
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    
    # Create the ring attractor network
    ringAttractor = RingAttractor(neuron_eq, 
                         num_neurons, 
                         Vth, V_reset, refractory_period,
                         syn_profile=syn_profile,
                         autapse=True,
                         sigma_exc=sigma_exc_val, 
                         sigma_inh=sigma_inh_val, 
                         g_exc=g_exc, 
                         g_inh=g_inh) 
    
    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    
    # Setup monitors
    spikemon = SpikeMonitor(ringAttractor.ring_pool)
    statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
    inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)
    
    # Network Operations:
    # Clipping - Reverse Potential Behaviour
    # Define a network operation to enforce the lower bound
    @network_operation(dt=defaultclock.dt)
    def enforce_lower_bound():
        # Using the built-in clip function (from numpy)
        ringAttractor.ring_pool.V[:] = clip(ringAttractor.ring_pool.V[:], V_reset, inf*volt)
    
    net = Network(ringAttractor.BrianObjects + [enforce_lower_bound, spikemon, statemon, inputmon])
    
    # Run simulation
    half_duration = sim_duration / 2
    net.run(half_duration)
    # Turn off input for the second half
    ringAttractor.ring_pool.I_ext = I_ext_array * 0
    net.run(half_duration)
    
    #+---------------------------------------------------------------------------+
    #|                           Plotting the Results                            |
    #+---------------------------------------------------------------------------+
    
    # Create a figure with multiple subplots using the modified plottingTools functions
    fig = plt.figure(figsize=(15, 15))
    
    # 1. Input Current Plot
    ax1 = fig.add_subplot(4, 2, 1)
    ax1.plot(positions/(2*pi), I_ext_array/mV)
    ax1.set_title('Input Current')
    ax1.set_xlabel('Position (rad)')
    ax1.set_ylabel('Current (mV)')
    
    # 2. TBD
    ax2 = fig.add_subplot(4, 2, 2)
    ax2.plot(positions/(2*pi), I_ext_array/mV)
    ax2.set_title('Input Current')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Current (mV)')
    
    # 3. Raster Plot
    ax3 = fig.add_subplot(4, 2, 3)
    raster_plot(spikemon, ax=ax3, stim_periods=(0,half_duration/second), stim_display_method='highlight', duration=sim_duration)
    
    # 3. Raster Plot
    ax4 = fig.add_subplot(4, 2, 4)
    raster_plot(spikemon, ax=ax4, stim_periods=(0,half_duration/second), stim_display_method='highlight', duration=sim_duration)

    # 4. Firing Rate Profile Plot
    ax4 = fig.add_subplot(4, 2, 5)
    firing_rate, _ = firing_rate_profile(spikemon, positions/(2*pi), sim_duration, ax=ax4)
    
    # 5. Polar Plot of the Population Vector Average (PVA)
    ax5 = fig.add_subplot(4, 2, 6, projection='polar')
    polar_plot_PVA(firing_rate, positions, scale=1.2, ax=ax5)

    # TODO:  Modify the function to highlight the sdcreen not the neurons
    # 6. Time-Resolved PVA Plot
    ax6 = fig.add_subplot(4, 2, 7)
    _, _ = time_resolved_PVA(spikemon, positions, sim_duration, num_neurons, ax=ax6, color_windows=True)

    # 7. Membrane potential traces
    ax7 = fig.add_subplot(4, 2, 8)
    membrane_potential_traces(statemon, sim_duration, ax=ax7)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Organize controls into two columns with equal number of widgets
left_column = VBox([
    num_neurons_slider,
    tau_slider,
    sigma_noise_slider,
    stimulus_center_slider,
    stimulus_width_slider,
    I0_slider
], layout=Layout(
    width='350px',
    justify_content='center'
))

# TODO: Modify the column based on the value of the profile dropdown if possible(?)
right_column = VBox([
    angles_dropdown,
    profile_dropdown,
    sigma_exc_slider,
    sigma_inh_slider,
    g_exc_slider,
    g_inh_slider,
    duration_slider
], layout=Layout(
    width='350px',
    justify_content='center'
))

# Arrange the two columns side by side
all_controls = HBox([
    left_column,
    right_column
], layout=Layout(
    margin='0 0 0 20px',  # Add margin on the left for spacing
    justify_content='center'  # Center the controls horizontally
))

# Create interactive output with a specific width
out = interactive_output(interactive_simulator, widgets)

# Create a horizontal box with output on the left and controls on the right
dashboard = HBox([
    out
], layout=Layout(
    align_items='center',  # Center items vertically
    justify_content='space-between'  # Distribute space between items
))

# Display the combined layout
display(all_controls)
display(dashboard)